# KNN Sınıflandırması — Iris Veri Seti

Bu notebook'ta K-En Yakın Komşu (KNN) algoritmasını Iris veri seti üzerinde uyguluyoruz.

**Hedef:** Çiçeğin ölçülerine göre türünü tahmin etmek (Setosa, Versicolor, Virginica)

## 1. Kütüphaneler

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## 2. Veri Seti

In [ ]:
iris = load_iris()

df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['species'] = iris.target
df['species_name'] = df['species'].map({0: 'Setosa', 1: 'Versicolor', 2: 'Virginica'})

print("Veri seti boyutu:", df.shape)
print("\nSınıf dağılımı:")
print(df['species_name'].value_counts())
df.head()

In [ ]:
df.describe()

## 3. Keşifsel Veri Analizi (EDA)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
features = iris.feature_names
colors = ['#e74c3c', '#3498db', '#2ecc71']
labels = ['Setosa', 'Versicolor', 'Virginica']

for idx, feature in enumerate(features):
    ax = axes[idx // 2][idx % 2]
    for i, (label, color) in enumerate(zip(labels, colors)):
        ax.hist(df[df['species'] == i][feature], bins=15, alpha=0.6, color=color, label=label)
    ax.set_title(feature)
    ax.set_xlabel('Değer (cm)')
    ax.set_ylabel('Frekans')
    ax.legend()

plt.suptitle('Özelliklerin Türlere Göre Dağılımı', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df[iris.feature_names].corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Özellikler Arası Korelasyon', fontsize=13)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Ön İşleme

In [ ]:
X = df[iris.feature_names].values
y = df['species'].values

# Min-Max Normalizasyon
scaler = MinMaxScaler()
X_normalized = scaler.fit_transform(X)

# Train-Test Split (%70 eğitim, %30 test)
X_train, X_test, y_train, y_test = train_test_split(
    X_normalized, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Eğitim seti: {X_train.shape[0]} örnek")
print(f"Test seti  : {X_test.shape[0]} örnek")

## 5. En İyi k Değerini Bulma

In [ ]:
k_values = range(1, 21)
train_scores = []
test_scores = []
cv_scores = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    train_scores.append(knn.score(X_train, y_train))
    test_scores.append(knn.score(X_test, y_test))
    cv = cross_val_score(knn, X_normalized, y, cv=5)
    cv_scores.append(cv.mean())

best_k = k_values[np.argmax(cv_scores)]
print(f"En iyi k değeri (5-fold CV): {best_k}")
print(f"CV doğruluğu: {max(cv_scores):.4f}")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(k_values, train_scores, 'o-', label='Eğitim', color='#3498db')
plt.plot(k_values, test_scores, 's-', label='Test', color='#e74c3c')
plt.plot(k_values, cv_scores, '^--', label='5-Fold CV', color='#2ecc71')
plt.axvline(x=best_k, color='gray', linestyle='--', alpha=0.7, label=f'En iyi k={best_k}')
plt.xlabel('k (Komşu Sayısı)')
plt.ylabel('Doğruluk')
plt.title('k Değerine Göre KNN Performansı')
plt.legend()
plt.xticks(k_values)
plt.ylim(0.85, 1.02)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('k_values_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Model Eğitimi ve Değerlendirme

In [ ]:
knn = KNeighborsClassifier(n_neighbors=best_k)
knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"Test Doğruluğu: {acc:.4f} ({acc*100:.2f}%)")
print(f"Doğru tahmin  : {np.sum(y_pred == y_test)} / {len(y_test)}")

In [ ]:
print("Sınıflandırma Raporu:")
print(classification_report(y_test, y_pred, target_names=labels))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels, linewidths=0.5)
plt.xlabel('Tahmin Edilen')
plt.ylabel('Gerçek')
plt.title(f'Confusion Matrix (k={best_k})', fontsize=13)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Sonuç

| Metrik | Değer |
|---|---|
| Algoritma | K-En Yakın Komşu (KNN) |
| Veri Seti | Iris (150 örnek, 4 özellik, 3 sınıf) |
| En iyi k | CV ile belirlendi |
| Normalizasyon | Min-Max Scaler |
| Test Doğruluğu | Notebook çalıştırıldığında görünür |